# 0. Configuration

In [41]:
# data load
from pathlib import Path # help with folder and file path 

# paths
import os 
# dataframe
import pandas as pd
import numpy as np
from functools import reduce

# data visualization
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Model
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import roc_auc_score, precision_recall_curve


In [20]:
# build paths

base_dir = os.getcwd()

# join parts of the path

def get_pd(folder,name):
    path = os.path.join('datasets',folder,name)
    return path

# 1. Load Data

In [34]:
X_train_path = get_pd('train','X_train.csv')
X_train = pd.read_csv(X_train_path)
X_train = X_train.iloc[:,1:]
X_train

,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,...,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions
0,False,2.0,1683.171527,461.778397,64.0,17.0,16.0,1.0,16.0,5.0,...,0.0,-999.0,1.0,0.0,1.0,76.04,1.0,-999.0,-999.0,2.319546
1,False,1.0,456.092951,150.224693,204.0,46.0,18.0,1.0,15.0,15.0,...,0.0,-999.0,1.0,0.0,1.0,20.69,-999.0,-999.0,-999.0,0.246326
2,False,-999.0,-999.000000,0.004687,64.0,17.0,7.0,-999.0,11.0,34.0,...,0.0,3.0,-999.0,0.0,2.0,-999.00,-999.0,1.0,1.0,-999.000000
3,Missing,-999.0,-999.000000,0.000000,386.0,17.0,8.0,-999.0,-999.0,-999.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
4,False,1.0,793.058791,338.916779,386.0,17.0,19.0,1.0,12.0,20.0,...,0.0,-999.0,1.0,0.0,1.0,38.27,1.0,-999.0,-999.0,11.731918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,Missing,-999.0,-999.000000,-999.000000,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
2633,False,1.0,368.115671,191.010846,612.0,17.0,11.0,1.0,12.0,5.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,4.483389
2634,False,1.0,314.989689,1672.900503,386.0,17.0,9.0,1.0,20.0,5.0,...,1.0,-999.0,1.0,0.0,1.0,28.96,-999.0,-999.0,-999.0,15.497548
2635,False,-999.0,-999.000000,723.849750,386.0,17.0,18.0,-999.0,13.0,33.0,...,0.0,-999.0,-999.0,0.0,2.0,-999.00,-999.0,-999.0,-999.0,-999.000000


In [32]:
y_train_path = get_pd('train','y_train.csv')
y_train = pd.read_csv(y_train_path).iloc[:,1:]
y_train

,is_fraudulent
0,0
1,0
2,0
3,0
4,0
...,...
2632,0
2633,0
2634,0
2635,0


In [36]:
X_val_path = get_pd('val','X_val.csv')
X_val = pd.read_csv(X_val_path).iloc[:,1:]
X_val

,latest_changed_password,max_item_count,days_since_first_transaction,email_address_age_days,email_domain_label,email_domain_tld_label,email_username_length,num_unique_delivery_addresses,latest_delivery_address_name_length,latest_delivery_address_region_label,...,per_week_payment_method_change,per_day_add_item_to_cart,per_day_transactions,per_day_payment_method_change,per_day_devices_per_user,per_day_purchase_total,per_day_unique_billing_last4,per_month_logout,per_month_page_activity,per_month_transactions
0,False,1.0,295.442852,178.913825,386.0,17.0,6.0,1.0,13.0,33.0,...,0.0,-999.0,1.0,0.0,1.0,18.58,-999.0,-999.0,-999.0,6.269334
1,False,1.0,290.811717,874.114088,386.0,17.0,13.0,1.0,14.0,27.0,...,0.0,-999.0,1.0,0.0,2.0,20.36,1.0,-999.0,1.0,4.483389
2,False,2.0,273.133769,1085.244366,1100.0,17.0,8.0,1.0,12.0,6.0,...,0.0,-999.0,-999.0,0.0,1.0,0.00,-999.0,-999.0,-999.0,4.483389
3,False,1.0,314.989689,389.277762,721.0,46.0,6.0,1.0,16.0,3.0,...,0.0,-999.0,1.0,0.0,1.0,28.28,-999.0,-999.0,-999.0,5.996951
4,Missing,-999.0,-999.000000,-999.000000,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,0.0,50.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,False,2.0,965.785600,332.593053,472.0,17.0,16.0,1.0,17.0,34.0,...,0.0,-999.0,1.0,0.0,1.0,36.49,-999.0,-999.0,-999.0,5.042047
656,False,1.0,827.610115,691.121393,322.0,28.0,15.0,1.0,15.0,8.0,...,0.0,-999.0,1.0,0.0,1.0,22.10,-999.0,-999.0,-999.0,6.248718
657,Missing,-999.0,-999.000000,-999.000000,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0,...,0.0,-999.0,-999.0,0.0,1.0,-999.00,-999.0,-999.0,-999.0,-999.000000
658,Missing,1.0,144.186578,1314.183530,386.0,17.0,14.0,1.0,13.0,18.0,...,0.0,-999.0,1.0,0.0,1.0,18.96,-999.0,-999.0,-999.0,3.817449


# 2. Data Preprocessing 

- categorical values encoding
- numerical values standardizing

In [39]:
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X_train.select_dtypes(include=["float64","int64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers = [
        # handle unknown category
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features),
        ("num",StandardScaler(), numerical_features),
        
    ]
)

# 3. Feature Selection

# 4. Baseline Model: Train & Test

In [43]:
# create XGB model

xgb_clf = XGBClassifier(
    objective='binary:logistic',
    n_estimators=30,
    learning_rate=0.1,
    max_depth=10,
    random_state=42,
    enable_categorical=True,
    use_label_encoder=False,
    eval_metric='logloss'
)


# create pipeline to train model
model = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",xgb_clf)
])

# fit model
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_train)
y_proba = model.predict_proba(X_train)[:,1]


/Users/chohasong/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [20:11:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# 5. Baseline Model: Evalation

In [ ]:
# on train 
roc_auc_score(y_train,y_proba)

0.9994726550650781

# References

https://www.ibm.com/think/tutorials/gradient-boosting-classifier#179371088

https://xgboost.readthedocs.io/en/release_3.2.0/get_started.html


https://feature-engine.trainindata.com/en/1.8.x/user_guide/encoding/WoEEncoder.html
